<a href="https://colab.research.google.com/github/alwaysalearner1234/ML03/blob/main/Sequence_Alignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
def score(a, b):
    if a == b:
        return 2      # match
    elif a == '-' or b == '-':
        return -2     # gap penalty
    else:
        return -1     # mismatch


In [2]:
def needleman_wunsch_matrix(A, B, score_fn):
    m, n = len(A), len(B)
    M = [[0]*(n+1) for _ in range(m+1)]

    # Initialization
    for i in range(1, m+1):
        M[i][0] = M[i-1][0] + score_fn(A[i-1], '-')
    for j in range(1, n+1):
        M[0][j] = M[0][j-1] + score_fn('-', B[j-1])

    # Fill DP matrix
    for i in range(1, m+1):
        for j in range(1, n+1):
            match = M[i-1][j-1] + score_fn(A[i-1], B[j-1])
            delete = M[i-1][j] + score_fn(A[i-1], '-')
            insert = M[i][j-1] + score_fn('-', B[j-1])
            M[i][j] = max(match, delete, insert)

    return M


In [3]:
def traceback_global(A, B, M, score_fn):
    i, j = len(A), len(B)
    alignA, alignB, matchline = [], [], []

    while i > 0 or j > 0:
        if i > 0 and j > 0 and M[i][j] == M[i-1][j-1] + score_fn(A[i-1], B[j-1]):
            alignA.append(A[i-1])
            alignB.append(B[j-1])
            matchline.append('|' if A[i-1] == B[j-1] else ' ')
            i -= 1
            j -= 1
        elif i > 0 and M[i][j] == M[i-1][j] + score_fn(A[i-1], '-'):
            alignA.append(A[i-1])
            alignB.append('-')
            matchline.append(' ')
            i -= 1
        else:
            alignA.append('-')
            alignB.append(B[j-1])
            matchline.append(' ')
            j -= 1

    return ''.join(reversed(alignB)), ''.join(reversed(matchline)), ''.join(reversed(alignA))


In [4]:
def semiglobal_matrix(A, B, score_fn):
    m, n = len(A), len(B)
    M = [[0]*(n+1) for _ in range(m+1)]

    # First column penalized, first row free
    for i in range(1, m+1):
        M[i][0] = M[i-1][0] + score_fn(A[i-1], '-')

    for i in range(1, m+1):
        for j in range(1, n+1):
            M[i][j] = max(
                M[i-1][j-1] + score_fn(A[i-1], B[j-1]),
                M[i-1][j] + score_fn(A[i-1], '-'),
                M[i][j-1] + score_fn('-', B[j-1])
            )

    return M


In [5]:
def smith_waterman_matrix(A, B, score_fn):
    m, n = len(A), len(B)
    M = [[0]*(n+1) for _ in range(m+1)]

    for i in range(1, m+1):
        for j in range(1, n+1):
            M[i][j] = max(
                0,
                M[i-1][j-1] + score_fn(A[i-1], B[j-1]),
                M[i-1][j] + score_fn(A[i-1], '-'),
                M[i][j-1] + score_fn('-', B[j-1])
            )
    return M


In [6]:
A = "RIDDLE"
B = "TRIPLE"

# Global Alignment
M = needleman_wunsch_matrix(A, B, score)
top, mid, bottom = traceback_global(A, B, M, score)

print("GLOBAL ALIGNMENT SCORE:", M[len(A)][len(B)])
print(top)
print(mid)
print(bottom)


GLOBAL ALIGNMENT SCORE: 3
TRI-PLE
 ||  ||
-RIDDLE
